# Chapter 16: Binary Search Trees

The programs from Chapter 16 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Searching and Inserting

`bst.py`

In [1]:
"""A binary search tree: every key in a node's left subtree is smaller
than the node's key, and every key in its right subtree is bigger.

All loops, no recursion: a badly unbalanced tree can be thousands of
levels deep, deeper than Python's recursion limit.
"""


class Node:
    __slots__ = ("key", "value", "left", "right")

    def __init__(self, key, value):
        self.key, self.value = key, value
        self.left = self.right = None


class BST:
    def __init__(self):
        self.root = None
        self._n = 0

    def __len__(self):
        return self._n

    def _find(self, key):
        """(node with key or None, its parent)."""
        parent, node = None, self.root
        while node and node.key != key:
            parent = node
            node = node.left if key < node.key else node.right
        return node, parent

    def get(self, key, default=None):
        node, _ = self._find(key)
        return node.value if node else default

    def __contains__(self, key):
        return self._find(key)[0] is not None

    def put(self, key, value):
        node, parent = self._find(key)
        if node:
            node.value = value  # the key is there: replace its value
        elif parent is None:
            self.root = Node(key, value)
            self._n += 1
        else:
            child = Node(key, value)
            if key < parent.key:
                parent.left = child
            else:
                parent.right = child
            self._n += 1

    def delete(self, key):
        node, parent = self._find(key)
        if node is None:
            raise KeyError(key)
        if node.left and node.right:
            # Two children: copy in the next key (the smallest on the
            # right), then delete that node, which has no left child
            parent, successor = node, node.right
            while successor.left:
                parent, successor = successor, successor.left
            node.key, node.value = successor.key, successor.value
            node = successor
        child = node.left or node.right  # at most one child now
        if parent is None:
            self.root = child
        elif parent.left is node:
            parent.left = child
        else:
            parent.right = child
        self._n -= 1

    def __iter__(self):
        """Keys in sorted order: an in-order walk with a stack."""
        stack, node = [], self.root
        while stack or node:
            while node:
                stack.append(node)
                node = node.left
            node = stack.pop()
            yield node.key
            node = node.right

    def min(self):
        node = self.root
        while node.left:
            node = node.left
        return node.key

    def floor(self, key):
        """The largest key <= key, or None."""
        node, best = self.root, None
        while node:
            if node.key == key:
                return key
            if node.key < key:
                best, node = node.key, node.right
            else:
                node = node.left
        return best

    def height(self):
        """Edges on the longest path down from the root, counted level
        by level."""
        level, h = [self.root] if self.root else [], -1
        while level:
            h += 1
            level = [c for n in level for c in (n.left, n.right) if c]
        return h

## Ordered Operations

`bst_demo.py`

In [2]:
"""Build a small BST, draw it, then delete the root."""

from bst import BST


def draw(node, depth=0):
    if node:
        draw(node.right, depth + 1)
        print("    " * depth + str(node.key))
        draw(node.left, depth + 1)


t = BST()
for k in [50, 30, 70, 20, 40, 60, 80]:
    t.put(k, None)
draw(t.root)
t.delete(50)  # 60, the next key, takes its place
print("after deleting 50:")
draw(t.root)

        80
    70
        60
50
        40
    30
        20
after deleting 50:
        80
    70
60
        40
    30
        20


## The Sorted-Input Disaster

`sorted_disaster.py`

In [3]:
"""The height of a BST built from random keys and from sorted keys."""

import math
import random

from bst import BST

rng = random.Random(2026)
print(f"{'n':>7} {'log2 n':>7} {'random order':>13} {'sorted order':>13}")
for n in [100, 1_000, 10_000]:
    shuffled, ordered = BST(), BST()
    for k in rng.sample(range(n), n):
        shuffled.put(k, None)
    for k in range(n):
        ordered.put(k, None)
    print(
        f"{n:>7,} {math.log2(n):>7.1f} {shuffled.height():>13,} "
        f"{ordered.height():>13,}"
    )

      n  log2 n  random order  sorted order
    100     6.6            14            99
  1,000    10.0            18           999


 10,000    13.3            26         9,999


`bench_bst.py`

In [4]:
"""Looking up a key: a BST built from random keys, one built from
sorted keys, and Python's dict."""

import random

import bench
from bst import BST

rng = random.Random(2026)


def built(order):
    def make(n):
        t = BST()
        for k in order(n):
            t.put(k, None)
        return t, n - 1  # the largest key: deepest in the sorted tree

    return make


def lookup(t, key):
    return key in t


sizes = [500, 1_000, 2_000, 4_000]
results = {
    "random keys": bench.doubling(
        lookup, built(lambda n: rng.sample(range(n), n)), sizes
    ),
    "sorted keys": bench.doubling(lookup, built(range), sizes),
    "dict": bench.doubling(
        lookup, lambda n: (dict.fromkeys(range(n)), n - 1), sizes
    ),
}
bench.table(results)
bench.chart(results, "ch16_bst.png", "Looking up the largest key")

         n     random keys     sorted keys            dict
       500          281 ns         14.3 us         66.1 ns
     1,000     277 ns x1.0    27.4 us x1.9    67.6 ns x1.0
     2,000     298 ns x1.1    56.9 us x2.1    79.4 ns x1.2
     4,000     379 ns x1.3     110 us x1.9    79.2 ns x1.0
